# Feature-importance significance testing



| procedure | question | statistic |
|---|---|---|
| Cluster bootstrap, 500 replicates | how much does the importance move when the protein sample changes? | Gini, mean and 95 % percentile interval |
| Altmann permutation null, 500 permutations | is it larger than the same forest assigns under random labels? | Gini, one-sided p, BH-adjusted across 84 |
| Out-of-fold permutation importance | does the feature help on data the model has not seen? | drop in AUROC (RFC) / MAE (RFR), mean and 95 % interval |
| Pairwise rank stability, top 20 | is the ordering between two features resolvable? | paired sign test, BH-adjusted across 190 pairs |

They disagree in informative ways, and a feature has to be read across all four rather than
by any one of them. Both the enrichment classifier (RFC) and the abundance regressor (RFR)
are run. The Altmann null needs 1000 forest refits, so the notebook takes about 20 minutes.

In [1]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Data, feature columns and RF settings shared with the other notebooks: corona.py.
from corona import (
    DATA_DIR,
    RANDOM_SEED,
    RandomForestClassifier,
    RandomForestRegressor,
    feature_cols,
    load_data,
    oof_permutation_importance,
    rf_kwargs,
)

N_PERM = 2000  # 1/(N_PERM + 1) must be <= 0.05/84 for a lone feature to be detectable
PERM_REPEATS = 10  # repeats per fold for out-of-fold permutation importance
N_FOLDS = 10
TOP_K = 20  # features entering the pairwise rank test

df = load_data()
feature_columns = feature_cols(df)

x = df[feature_columns].values
proteins = df["entry"].values
TARGETS = {
    "RFC": (RandomForestClassifier, df["enriched"].values, "roc_auc"),
    "RFR": (RandomForestRegressor, np.log1p(df["abundance"].values), "neg_mean_absolute_error"),
}

print(f"n={len(df)}, features={len(feature_columns)}, proteins={df['entry'].nunique()}")

n=6768, features=84, proteins=376


## Observed importance and bootstrap stability

Baseline Gini importance comes from a single fit on all rows. The cluster-bootstrap
replicates resample whole proteins, preserving the within-protein correlation structure, and
are read from the replicates `unified.ipynb` already produced rather than refitted, so the
intervals match those replicates exactly.

In [2]:
observed, bootstrap = {}, {}
for name, (Model, y, _) in TARGETS.items():
    observed[name] = pd.Series(
        Model(**rf_kwargs).fit(x, y).feature_importances_, index=feature_columns
    )
    raw = pd.read_csv(
        DATA_DIR / f"{name.lower()}_bootstrap_importance_raw.csv"
    ).drop(columns="bootstrap_iter")[feature_columns]
    bootstrap[name] = raw
    print(f"{name}: observed Gini for {len(observed[name])} features, "
          f"bootstrap {raw.shape[0]} replicates")

RFC: observed Gini for 84 features, bootstrap 500 replicates


RFR: observed Gini for 84 features, bootstrap 500 replicates


## Altmann permutation null

Permuting the target and refitting gives the importance the same forest assigns to each
feature when the labels carry no information. The one-sided p-value is
`(1 + #{null >= observed}) / (1 + N_perm)`.

That formula puts a floor on the p-value at `1/(N_perm + 1)`, and the floor has to be low
enough for the multiplicity correction to clear it. Benjamini-Hochberg calls the *k*-th
smallest p-value significant only if it falls below *k*/m x alpha, so a feature that is
significant on its own must reach alpha/m = 0.05/84 = 0.00060. At 500 permutations the floor
is 0.00200, three times too coarse: significance becomes reachable only when four or more
features happen to sit on the floor together and lift the threshold above it. At 2000 the
floor is 0.00050 and a single feature can be resolved. The general requirement is
`1/(N_perm + 1) <= alpha/m`, here N_perm >= 1679.

This is the expensive cell: 4000 forest refits, dominated by the regressor.

In [3]:
altmann = {}
for name, (Model, y, _) in TARGETS.items():
    rng = np.random.default_rng(RANDOM_SEED)
    null = np.empty((N_PERM, len(feature_columns)))
    for k in range(N_PERM):
        model = Model(**{**rf_kwargs, "random_state": RANDOM_SEED + 10_000 + k})
        null[k] = model.fit(x, rng.permutation(y)).feature_importances_
        if (k + 1) % 250 == 0:
            print(f"  {name}: permutation {k + 1}/{N_PERM}")
    null_df = pd.DataFrame(null, columns=feature_columns)
    p = (null_df.ge(observed[name], axis=1).sum(axis=0) + 1) / (N_PERM + 1)
    altmann[name] = pd.DataFrame(
        {
            "null_mean": null_df.mean(),
            "null_q95": null_df.quantile(0.95),
            "p_altmann": p,
            "p_altmann_bh": multipletests(p.values, method="fdr_bh")[1],
        }
    )
    print(f"{name}: {(altmann[name].p_altmann_bh < 0.05).sum()}/{len(feature_columns)} features pass BH at 5%")

  RFC: permutation 250/2000


  RFC: permutation 500/2000


  RFC: permutation 750/2000


  RFC: permutation 1000/2000


  RFC: permutation 1250/2000


...

  RFR: permutation 1000/2000


  RFR: permutation 1250/2000


  RFR: permutation 1500/2000


  RFR: permutation 1750/2000


  RFR: permutation 2000/2000
RFR: 3/84 features pass BH at 5%


In [4]:
# The features the tests turn on: those the regressor resolves, and the continuous features
# whose null exceeds their observed importance.
FOCUS = [
    "abundance_controls",
    "nsp_secondary_structure_coil",
    "nsp_secondary_structure_helix",
    "secondary_structure_fraction_sheet",
    "zeta_potential",
    "dh_functionalized",
    "dtem",
]

altmann_view = (
    pd.concat(
        {n: altmann[n].assign(gini_observed=observed[n]) for n in TARGETS},
        names=["model", "feature"],
    )
    .reset_index()
    .loc[:, ["model", "feature", "gini_observed", "null_mean", "p_altmann", "p_altmann_bh"]]
)
altmann_view["significant"] = altmann_view.p_altmann_bh < 0.05

for name in TARGETS:
    sig = altmann_view.query("model == @name and significant")
    print(f"{name}: {len(sig)}/{len(feature_columns)} significant -> {', '.join(sig.feature) if len(sig) <= 5 else str(len(sig)) + ' features'}")

altmann_view[altmann_view.feature.isin(FOCUS)].round(5)

RFC: 32/84 significant -> 32 features
RFR: 3/84 significant -> abundance_controls, nsp_secondary_structure_coil, nsp_secondary_structure_helix


,model,feature,gini_observed,null_mean,p_altmann,p_altmann_bh,significant
0,RFC,abundance_controls,0.02920,0.00364,0.0005,0.00300,True
33,RFC,secondary_structure_fraction_sheet,0.02396,0.00966,0.0005,0.00300,True
65,RFC,nsp_secondary_structure_coil,0.01235,0.00928,0.0020,0.00933,True
67,RFC,nsp_secondary_structure_helix,0.01515,0.00950,0.0005,0.00300,True
69,RFC,zeta_potential,0.04708,0.06279,1.0000,1.00000,False
79,RFC,dtem,0.01743,0.04688,1.0000,1.00000,False
80,RFC,dh_functionalized,0.02526,0.07062,1.0000,1.00000,False
84,RFR,abundance_controls,0.67381,0.00516,0.0005,0.02099,True
117,RFR,secondary_structure_fraction_sheet,0.00236,0.00683,1.0000,1.00000,False
149,RFR,nsp_secondary_structure_coil,0.01935,0.00601,0.0005,0.02099,True


Both models separate from their nulls, but by very different margins. The classifier clears
BH at 5 % for 32 of 84 features, six of the seven in the parsimonious set. The regressor
clears 3: `abundance_controls` (observed Gini 0.674 against a null mean of 0.005),
`nsp_secondary_structure_coil` and `nsp_secondary_structure_helix`. That asymmetry is the
shape of the two models. Abundance regression is carried almost entirely by one feature,
which takes two thirds of the total impurity reduction, while enrichment classification
spreads its signal across many weak protein descriptors.

`zeta_potential` fails in **both** models at p = 1.000, and the reason is visible in the null
itself: its null mean sits *above* its observed importance, 0.063 against 0.047 for the
classifier and 0.104 against 0.023 for the regressor. Impurity importance rewards features
that offer many split points, so a continuous variable accumulates importance even when the
labels are random. The test cannot detect signal in a continuous feature, and a failure here
carries no information about it either way. The same applies to `dh_functionalized` and
`dtem`. For those features the out-of-fold permutation importance is the measure that
discriminates, because shuffling a column and re-scoring held-out data does not reward split
count. `zeta_potential` is the one member of the parsimonious set the Altmann test rejects,
and it is rejected for being continuous rather than for being uninformative.

## Out-of-fold permutation importance

Ten folds, ten repeats per fold, giving 100 measurements per feature: the drop in held-out
performance when one column is shuffled.

Both models are grouped by protein here, including the regressor, which is grouped by
nanoparticle everywhere else. The reason is structural rather than conventional. Permutation
importance shuffles a column *within* the held-out fold, so a feature that is constant inside
that fold cannot be permuted into anything different and scores zero however much the model
relies on it. Holding out one nanoparticle makes all twelve nanoparticle descriptors constant
in the test fold, which would report zeta potential, core, ligand and diameter as exactly zero
by construction. Holding out proteins leaves every one of the 84 features varying within the
fold, so all of them remain measurable. Since these values are what the feature ranking is
built from, and the parsimonious set contains a nanoparticle feature, nanoparticle grouping
would zero out a feature the model keeps.


In [5]:
# `oof_permutation_importance` (corona.py): the same folds, seeds and repeats as
# `unified.ipynb`, so the RFC rows reproduce its `classifier_permutation_importance.csv`.
oof_perm = {}
for name, (Model, y, scoring) in TARGETS.items():
    print(name)
    vals = oof_permutation_importance(
        Model, df[feature_columns], y, proteins, scoring, N_FOLDS, PERM_REPEATS
    )
    oof_perm[name] = pd.DataFrame(
        {
            "perm_mean": vals.mean(),
            "perm_ci_low": vals.quantile(0.025),
            "perm_ci_high": vals.quantile(0.975),
            "perm_ci_excludes_zero": vals.quantile(0.025) > 0,
        }
    )
    print(f"{name}: {oof_perm[name].perm_ci_excludes_zero.sum()}/{len(feature_columns)} features with CI above zero")

RFC


  fold 1/10


  fold 2/10


  fold 3/10


  fold 4/10


...

  fold 6/10


  fold 7/10


  fold 8/10


  fold 9/10


  fold 10/10
RFR: 1/84 features with CI above zero


In [6]:
# Top five by mean held-out importance per model, with the interval that decides
# `perm_ci_excludes_zero`.
perm_view = (
    pd.concat({n: oof_perm[n] for n in TARGETS}, names=["model", "feature"])
    .reset_index()
    .sort_values(["model", "perm_mean"], ascending=[True, False])
)
perm_view.groupby("model").head(5).round(5)

,model,feature,perm_mean,perm_ci_low,perm_ci_high,perm_ci_excludes_zero
0,RFC,abundance_controls,0.02033,-0.02688,0.05957,False
33,RFC,secondary_structure_fraction_sheet,0.01860,0.00007,0.05043,True
69,RFC,zeta_potential,0.01683,-0.00071,0.03160,False
76,RFC,ligand_pei,0.01087,-0.00034,0.02522,False
65,RFC,nsp_secondary_structure_coil,0.00841,-0.00456,0.02430,False
84,RFR,abundance_controls,0.06974,0.02111,0.15489,True
118,RFR,secondary_structure_fraction_disordered,0.00154,-0.00154,0.00662,False
91,RFR,frac_aa_f,0.00086,-0.00084,0.00308,False
132,RFR,fraction_exposed_exposed_e,0.00085,-0.00005,0.00355,False
149,RFR,nsp_secondary_structure_coil,0.00083,-0.00086,0.00313,False


One feature per model has a 95 % interval strictly above zero:
`secondary_structure_fraction_sheet` for the classifier and `abundance_controls` for the
regressor. Treating "interval excludes zero" as the bar for contributing would leave a single
contributing feature per model, which is too strict a reading of this interval.

The percentile spans 100 measurements that mix ten folds with ten repeats inside each fold,
and between-fold variation dominates. It describes how far the importance moves between
held-out sets of proteins, not how far the mean sits from zero. Read as a stability measure
it is informative; read as a significance test it is far too conservative, which is why the
permutation *mean* rather than its interval is the useful column here.

## Combined table

One row per feature per model: observed importance, bootstrap interval, Altmann null and
p-values, and out-of-fold permutation importance.

In [7]:
supplementary = pd.concat(
    {
        name: pd.concat(
            [
                observed[name].rename("gini_observed"),
                bootstrap[name].mean().rename("gini_boot_mean"),
                bootstrap[name].quantile(0.025).rename("gini_boot_ci_low"),
                bootstrap[name].quantile(0.975).rename("gini_boot_ci_high"),
                altmann[name],
                oof_perm[name],
            ],
            axis=1,
        )
        for name in TARGETS
    },
    names=["model", "feature"],
).reset_index()
supplementary = supplementary.sort_values(
    ["model", "gini_observed"], ascending=[True, False]
).reset_index(drop=True)

print("RFC, top 15 by observed Gini:")
supplementary.query("model == 'RFC'").head(15).round(5)

RFC, top 15 by observed Gini:


,model,feature,gini_observed,gini_boot_mean,gini_boot_ci_low,gini_boot_ci_high,null_mean,null_q95,p_altmann,p_altmann_bh,perm_mean,perm_ci_low,perm_ci_high,perm_ci_excludes_zero
0,RFC,zeta_potential,0.04708,0.06053,0.05250,0.06801,0.06279,0.06638,1.0000,1.00000,0.01683,-0.00071,0.03160,False
1,RFC,abundance_controls,0.02920,0.02077,0.00853,0.03783,0.00364,0.00480,0.0005,0.00300,0.02033,-0.02688,0.05957,False
2,RFC,dh_functionalized,0.02526,0.03423,0.02948,0.03877,0.07062,0.07444,1.0000,1.00000,0.00503,-0.00034,0.01466,False
3,RFC,secondary_structure_fraction_sheet,0.02396,0.01996,0.01032,0.03269,0.00966,0.01094,0.0005,0.00300,0.01860,0.00007,0.05043,True
4,RFC,ligand_pei,0.02358,0.03298,0.02692,0.03906,0.01807,0.01975,0.0005,0.00300,0.01087,-0.00034,0.02522,False
5,RFC,dtem,0.01743,0.02229,0.01883,0.02622,0.04688,0.04997,1.0000,1.00000,0.00529,-0.00102,0.01031,False
6,RFC,frac_aa_a,0.01536,0.01568,0.00876,0.02812,0.01020,0.01157,0.0005,0.00300,0.00591,-0.00804,0.03111,False
7,RFC,fraction_exposed,0.01522,0.01177,0.00759,0.01862,0.00845,0.00961,0.0005,0.00300,0.00385,-0.00634,0.01599,False
8,RFC,nsp_secondary_structure_helix,0.01515,0.01287,0.00836,0.02066,0.00950,0.01070,0.0005,0.00300,0.00239,-0.00883,0.01431,False
9,RFC,frac_aa_c,0.01484,0.01248,0.00771,0.02087,0.01032,0.01187,0.0005,0.00300,0.00566,-0.00201,0.01846,False


In [8]:
print("RFR, top 15 by observed Gini:")
supplementary.query("model == 'RFR'").head(15).round(5)

RFR, top 15 by observed Gini:


,model,feature,gini_observed,gini_boot_mean,gini_boot_ci_low,gini_boot_ci_high,null_mean,null_q95,p_altmann,p_altmann_bh,perm_mean,perm_ci_low,perm_ci_high,perm_ci_excludes_zero
84,RFR,abundance_controls,0.67381,0.62359,0.38925,0.81509,0.00516,0.00959,0.00050,0.02099,0.06974,0.02111,0.15489,True
85,RFR,zeta_potential,0.02326,0.02991,0.01311,0.05898,0.10361,0.11898,1.00000,1.00000,0.00067,-0.00090,0.00213,False
86,RFR,nsp_secondary_structure_coil,0.01935,0.00472,0.00053,0.02222,0.00601,0.00855,0.00050,0.02099,0.00083,-0.00086,0.00313,False
87,RFR,nsp_secondary_structure_helix,0.01775,0.00562,0.00062,0.02620,0.00664,0.00933,0.00100,0.02799,0.00079,-0.00091,0.00330,False
88,RFR,dh_functionalized,0.01477,0.01517,0.00983,0.02235,0.13564,0.15041,1.00000,1.00000,0.00035,-0.00041,0.00127,False
89,RFR,frac_aa_r,0.01034,0.00629,0.00057,0.03374,0.00783,0.01105,0.07996,1.00000,0.00018,-0.00190,0.00219,False
90,RFR,incubation_concentration_mg_per_ml,0.00790,0.01181,0.00520,0.02094,0.04935,0.05717,1.00000,1.00000,0.00052,-0.00048,0.00153,False
91,RFR,frac_aa_n,0.00747,0.00451,0.00068,0.01930,0.00850,0.01229,0.66367,1.00000,0.00031,-0.00025,0.00094,False
92,RFR,frac_aa_m,0.00664,0.00494,0.00068,0.02040,0.00941,0.01335,0.93503,1.00000,-0.00012,-0.00113,0.00116,False
93,RFR,fraction_exposed_exposed_c,0.00663,0.00360,0.00067,0.01329,0.00806,0.01210,0.73413,1.00000,0.00014,-0.00062,0.00114,False


## Pairwise rank stability

Each bootstrap replicate refits on the same resampled proteins, so the difference between
two features' importances is paired within a replicate. The one-sided p-value is the
fraction of replicates whose difference contradicts the observed ordering, doubled for a
two-sided test and adjusted across the 190 pairs.

In [9]:
rank_rows = []
for name in TARGETS:
    top = observed[name].sort_values(ascending=False).head(TOP_K).index.tolist()
    boot = bootstrap[name][top].to_numpy()
    pairs = []
    for a in range(TOP_K):
        for b in range(a + 1, TOP_K):
            diff = boot[:, a] - boot[:, b]
            obs = observed[name][top[a]] - observed[name][top[b]]
            p_one = float(np.mean(diff <= 0) if obs >= 0 else np.mean(diff >= 0))
            pairs.append(
                {
                    "model": name,
                    "feature_higher": top[a],
                    "feature_lower": top[b],
                    "observed_diff": obs,
                    "p_two_sided": min(1.0, 2 * p_one),
                }
            )
    frame = pd.DataFrame(pairs)
    frame["p_bh"] = multipletests(frame.p_two_sided.values, method="fdr_bh")[1]
    frame["ordering_resolved"] = frame.p_bh < 0.05
    rank_rows.append(frame)
    print(f"{name}: {frame.ordering_resolved.sum()}/{len(frame)} of the "
          f"{TOP_K}-feature pairs have a resolvable ordering")

rank_stability = pd.concat(rank_rows, ignore_index=True)
# Five pairs per model rather than the first ten, which were all classifier rows.
rank_stability.groupby('model').head(5).round(5)

RFC: 54/190 of the 20-feature pairs have a resolvable ordering
RFR: 25/190 of the 20-feature pairs have a resolvable ordering


,model,feature_higher,feature_lower,observed_diff,p_two_sided,p_bh,ordering_resolved
0,RFC,zeta_potential,abundance_controls,0.01787,0.0,0.0,True
1,RFC,zeta_potential,dh_functionalized,0.02181,0.0,0.0,True
2,RFC,zeta_potential,secondary_structure_fraction_sheet,0.02312,0.0,0.0,True
3,RFC,zeta_potential,ligand_pei,0.02350,0.0,0.0,True
4,RFC,zeta_potential,dtem,0.02965,0.0,0.0,True
190,RFR,abundance_controls,zeta_potential,0.65055,0.0,0.0,True
191,RFR,abundance_controls,nsp_secondary_structure_coil,0.65446,0.0,0.0,True
192,RFR,abundance_controls,nsp_secondary_structure_helix,0.65605,0.0,0.0,True
193,RFR,abundance_controls,dh_functionalized,0.65904,0.0,0.0,True
194,RFR,abundance_controls,frac_aa_r,0.66347,0.0,0.0,True


Most of the ordering is noise: 54 of 190 pairs resolve for the classifier and 25 of 190 for
the regressor. The pairs that do resolve are mostly those involving the one dominant feature
in each model, `zeta_potential` and `abundance_controls` respectively, which separate from
everything below them.

Within the rest of the top 20 the ordering is not supported by the data. The importances of
those features should be treated as a set of comparable magnitude rather than as a ranking,
and small differences in rank between them carry no information.

In [10]:
supplementary.to_csv(DATA_DIR / "feature_importance_supplementary.csv", index=False)
rank_stability.to_csv(DATA_DIR / "importance_rank_stability.csv", index=False)
print("wrote 2 tables to", DATA_DIR)

wrote 2 tables to data
